In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import sys
sys.path.append('/WORK/Codes/global_lake_area/my_unet_gdal/')
import area_calculation

In [2]:
lake_lse_gdf_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data.pkl'
lake_lse_gdf = pd.read_pickle(lake_lse_gdf_path)
lake_lse_gdf_save_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data_and_unbuffered_boundary_area.pkl'
hybas_id_list = [
    1020000010, 1020011530, 1020018110, 1020021940, 1020027430, 1020034170, 1020035180, 1020040190,
    2020000010, 2020003440, 2020018240, 2020024230, 2020033490, 2020041390, 2020057170, 2020065840, 2020071190,
    3020000010, 3020003790, 3020005240, 3020008670, 3020009320, 3020024310,
    4020000010, 4020006940, 4020015090, 4020024190, 4020034510, 4020050210, 4020050220, 4020050290, 4020050470,
    5020000010, 5020015660, 5020037270, 5020049720, 5020082270, 
    6020000010, 6020006540, 6020008320, 6020014330, 6020017370, 6020021870, 6020029280,
    7020000010, 7020014250, 7020021430, 7020024600, 7020038340, 7020046750, 7020047840, 7020065090,
    8020000010, 8020008900, 8020010700, 8020020760, 8020022890, 8020032840, 8020044560,
    9020000010
]

lake_boundary_in_4326_proj_path_pattern = '/WORK/Data/global_lake_area/lake_shps/HydroLAKES_updated_using_GLAKES/per_basin_no_contained_new/hylak_unbuffered_updated_no_contained_{basin_id}.shp'
raster_to_provide_src_path_pattern = '/WORK/Data/global_lake_area/gsw_images/mosaic/{basin_id}/{basin_id}_gsw_30m_2001-01-01_2001-02-01.tif'
lake_boundary_in_ea_proj_path_pattern = '/WORK/Data/global_lake_area/lake_shps/HydroLAKES_updated_using_GLAKES/per_basin_no_contained_new/hylak_unbuffered_updated_no_contained_{basin_id}_reprojected.shp'
lake_lse_gdf['lake_boundary_area_unbuffered'] = np.nan

for hybas_id in hybas_id_list:
    print(f'Processing basin {hybas_id}')
    lake_boundary_in_4326_proj_path = lake_boundary_in_4326_proj_path_pattern.format(basin_id=hybas_id)
    lake_boundary_in_4326_proj_gdf = gpd.read_file(lake_boundary_in_4326_proj_path)
    raster_to_provide_src_path = raster_to_provide_src_path_pattern.format(basin_id=hybas_id)
    lake_boundary_in_ea_proj_path = lake_boundary_in_ea_proj_path_pattern.format(basin_id=hybas_id)
    reprojected = area_calculation.check_and_reproject_vector_srs_to_raster(
            raster_path=raster_to_provide_src_path,
            vector_path=lake_boundary_in_4326_proj_path,
            output_path=lake_boundary_in_ea_proj_path,
            verbose=1
        )
    lake_boundary_in_ea_proj_gdf = gpd.read_file(lake_boundary_in_ea_proj_path).set_index('Hylak_id')
    lake_boundary_in_ea_proj_gdf['lake_boundary_area_unbuffered'] = lake_boundary_in_ea_proj_gdf['geometry'].area
    lake_lse_gdf.update(lake_boundary_in_ea_proj_gdf['lake_boundary_area_unbuffered'])

lake_lse_gdf.to_pickle(lake_lse_gdf_save_path)


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1538: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


Processing basin 1020000010
Spatial references do not match, reprojecting...
Reprojected vector layer saved to: /WORK/Data/global_lake_area/lake_shps/HydroLAKES_updated_using_GLAKES/per_basin_no_contained_new/hylak_unbuffered_updated_no_contained_1020000010_reprojected.shp
Processing basin 1020011530
Spatial references do not match, reprojecting...
Reprojected vector layer saved to: /WORK/Data/global_lake_area/lake_shps/HydroLAKES_updated_using_GLAKES/per_basin_no_contained_new/hylak_unbuffered_updated_no_contained_1020011530_reprojected.shp
Processing basin 1020018110
Spatial references do not match, reprojecting...
Reprojected vector layer saved to: /WORK/Data/global_lake_area/lake_shps/HydroLAKES_updated_using_GLAKES/per_basin_no_contained_new/hylak_unbuffered_updated_no_contained_1020018110_reprojected.shp
Processing basin 1020021940
Spatial references do not match, reprojecting...
Reprojected vector layer saved to: /WORK/Data/global_lake_area/lake_shps/HydroLAKES_updated_using_GLA

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('/WORK/Codes/global_lake_area/my_spatial_analyze/')
import grid_analyze

lake_lse_gdf_with_unbuffered_boundary_area_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data_and_unbuffered_boundary_area.pkl'
lake_lse_gdf_with_unbuffered_boundary_area = pd.read_pickle(lake_lse_gdf_with_unbuffered_boundary_area_path).reset_index()
lake_lse_gdf_with_unbuffered_boundary_area['lake_boundary_area_unbuffered'] = lake_lse_gdf_with_unbuffered_boundary_area['lake_boundary_area_unbuffered'] / 1e6
lake_lse_gdf_with_unbuffered_boundary_area['mean_total_variation'] = lake_lse_gdf_with_unbuffered_boundary_area['mean_seasonal_amplitude'] + lake_lse_gdf_with_unbuffered_boundary_area['annual_means_std']
lake_lse_gdf_with_unbuffered_boundary_area['total_variation_relative_to_total_area'] = lake_lse_gdf_with_unbuffered_boundary_area['mean_total_variation'] / lake_lse_gdf_with_unbuffered_boundary_area['lake_boundary_area_unbuffered'] * 100

lake_lse_gdf_with_unbuffered_boundary_area_save_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data_and_unbuffered_boundary_area_and_total_variation_relative_to_total_area.pkl'
lake_lse_gdf_with_unbuffered_boundary_area.to_pickle(lake_lse_gdf_with_unbuffered_boundary_area_save_path)



Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
sys.path.append('/WORK/Codes/global_lake_area/my_spatial_analyze/')
import grid_analyze

lake_lse_gdf_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data_and_unbuffered_boundary_area_and_total_variation_relative_to_total_area.pkl'
lake_lse_gdf = pd.read_pickle(lake_lse_gdf_path)

area_start_date = '2001-01-01'
area_end_date = '2024-01-01'
date_fmt = '%Y-%m-%d'
area_start_date = datetime.strptime(area_start_date, date_fmt)
area_end_date = datetime.strptime(area_end_date, date_fmt)
area_columns = []
current_date = area_start_date
while current_date < area_end_date:
    area_columns.append(current_date.strftime(date_fmt))
    current_date = current_date + relativedelta(months=1)

lake_lse_gdf = lake_lse_gdf[['Hylak_id', 'lake_boundary_area_unbuffered', 'total_variation_relative_to_total_area'] + area_columns + ['centroid', 'geometry', 'Lake_type']]

additional_agg_dict_for_generating_grid = {
    'total_variation_relative_to_total_area': ('mean', 'median')
}

grid_gdf = grid_analyze.generate_grid_from_geometry_added_concatenated_areas(
    geometry_added_concatenated_areas_gdf=lake_lse_gdf,
    grid_size=0.5,
    area_columns=area_columns,
    grid_extent=[-180, -90, 180, 90],
    geometry_to_use_column='geometry',
    additional_agg_dict=additional_agg_dict_for_generating_grid,
    bootstrap_ks_test_column_pairs=None,
    bootstrap_ks_test_output_columns=None,
    verbose=1
)
grid_gdf = grid_gdf[grid_gdf['lake_count'] != 0]
grid_gdf_save_path = '/WORK/Data/global_lake_area/area_csvs/grids/pkl/grid_all_medium_for_relative_to_total_area.pkl'
grid_gdf.to_pickle(grid_gdf_save_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.
Generating grid cells...
Extent: (-180, -90, 180, 90)
Performing spatial join...
Aggregating data...
{'2001-01-01': 'sum', '2001-02-01': 'sum', '2001-03-01': 'sum', '2001-04-01': 'sum', '2001-05-01': 'sum', '2001-06-01': 'sum', '2001-07-01': 'sum', '2001-08-01': 'sum', '2001-09-01': 'sum', '2001-10-01': 'sum', '2001-11-01': 'sum', '2001-12-01': 'sum', '2002-01-01': 'sum', '2002-02-01': 'sum', '2002-03-01': 'sum', '2002-04-01': 'sum', '2002-05-01': 'sum', '2002-06-01': 'sum', '2002-07-01': 'sum', '2002-08-01': 'sum', '2002-09-01': 'sum', '2002-10-01': 'sum', '2002-11-01': 'sum', '2002-12-01': 'sum', '2003-01-01': 'sum', '2003-02-01': 'sum', '2003-03-01': 'sum', '2003-04-01': 'sum', '2003-05-01': 'sum', '2003-06-01': 'sum', '2003-07-01': 'sum', '2003-08-01': 'sum', '2003-09-01': 'sum', '2003-1

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import sys
sys.path.append('../../my_spatial_analyze/data_analyze/basin_wise_analysis')
import basin_wise_analysis

calculated_hydrobasins_pkl_path = '/WORK/Data/global_lake_area/hydrobasins/merged/basinatlas_lev06_with_total_variation_relative_to_total_area_for_revision.pkl'

basinatlas_shp_path = '/WORK/Data/global_lake_area/hydroATLAS/shp/BasinATLAS_v10_shp/BasinATLAS_v10_lev06.shp'
lake_lse_pkl_path = '/WORK/Data/global_lake_area/area_csvs/lakes/pkl/lakes_all_with_all_additional_attributes_revision_with_missing_data_and_unbuffered_boundary_area_and_total_variation_relative_to_total_area.pkl'
basinatlas_gdf = gpd.read_file(basinatlas_shp_path)
print(basinatlas_gdf.columns.tolist())
lake_lse_gdf = pd.read_pickle(lake_lse_pkl_path)[['Hylak_id', 'total_variation_relative_to_total_area', 'Pour_lat', 'Pour_long']]
lake_lse_lat_column = 'Pour_lat'
lake_lse_lon_column = 'Pour_long'
lake_lse_gdf['geometry'] = [Point(xy) for xy in zip(lake_lse_gdf[lake_lse_lon_column], lake_lse_gdf[lake_lse_lat_column])]
lake_lse_gdf = gpd.GeoDataFrame(lake_lse_gdf, crs='EPSG:4326')
#project hydrobasins to lake_lse_gdf's crs if not the same
if lake_lse_gdf.crs != basinatlas_gdf.crs:
    basinatlas_gdf = basinatlas_gdf.to_crs(lake_lse_gdf.crs)
column_names_and_statistics_to_calculate = {
    'total_variation_relative_to_total_area': 'median',
}

for column_name, statistics_type in column_names_and_statistics_to_calculate.items():
    basinatlas_gdf = basin_wise_analysis.add_basin_wise_statistics_to_hydrobasins(
        hydrobasins_gdf=basinatlas_gdf,
        lake_lse_gdf=lake_lse_gdf,
        column_name_to_calculate=column_name,
        statistics_type=statistics_type
    )
    
basinatlas_gdf.to_pickle(calculated_hydrobasins_pkl_path)

Initializing pandarallel...
INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.
['HYBAS_ID', 'NEXT_DOWN', 'NEXT_SINK', 'MAIN_BAS', 'DIST_SINK', 'DIST_MAIN', 'SUB_AREA', 'UP_AREA', 'PFAF_ID', 'ENDO', 'COAST', 'ORDER_', 'SORT', 'dis_m3_pyr', 'dis_m3_pmn', 'dis_m3_pmx', 'run_mm_syr', 'inu_pc_smn', 'inu_pc_umn', 'inu_pc_smx', 'inu_pc_umx', 'inu_pc_slt', 'inu_pc_ult', 'lka_pc_sse', 'lka_pc_use', 'lkv_mc_usu', 'rev_mc_usu', 'dor_pc_pva', 'ria_ha_ssu', 'ria_ha_usu', 'riv_tc_ssu', 'riv_tc_usu', 'gwt_cm_sav', 'ele_mt_sav', 'ele_mt_uav', 'ele_mt_smn', 'ele_mt_smx', 'slp_dg_sav', 'slp_dg_uav', 'sgr_dk_sav', 'clz_cl_smj', 'cls_cl_smj', 'tmp_dc_syr', 'tmp_dc_uyr', 'tmp_dc_smn', 'tmp_dc_smx', 'tmp_dc_s01', 'tmp_dc_s02', 'tmp_dc_s03', 'tmp_dc_s04', 'tmp_dc_s05', 'tmp_dc_s06', 'tmp_dc_s07', 'tmp_dc_s08', 'tmp_dc_s09', 'tmp_dc_s10', 'tmp_dc_s11', 'tmp_dc_s12', 'pre_mm_syr', 'pre_mm_uyr'

/usr/local/lib/python3.8/dist-packages/IPython/core/interactiveshell.py:3508: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


HYBAS_ID
1060000160    3.313623
1060001090    7.364486
1060001380    1.881300
1060001520    3.760768
1060002300    3.489576
                ...   
9060031700    0.000000
9060081990    0.238353
9060086030         NaN
9060109730    0.264575
9060111620         NaN
Name: total_variation_relative_to_total_area_median, Length: 11371, dtype: float64
